In [7]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

torch.backends.cudnn.benchmark = True 
 
%matplotlib inline

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [11]:
DATA_DIR = "data/art_portraits/Portraits_update"
image_size = 32
batch_size = 128

stats = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

full_ds = ImageFolder(
    DATA_DIR,
    transform=T.Compose([
        T.Resize(image_size),
        T.CenterCrop(image_size),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(*stats)
    ])
)


np.random.seed(42)
indices = np.random.choice(len(full_ds), size=3000, replace=False)
train_ds = Subset(full_ds, indices)

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,        
    pin_memory=True
)

print("Total images:", len(train_ds))

Total images: 3000


In [12]:
x, _ = next(iter(train_dl))
x = x.to(device)
print(x.shape, x.min().item(), x.max().item())


torch.Size([128, 3, 32, 32]) -1.0 1.0


In [13]:
class VanillaGenerator(nn.Module):
    def __init__(self, nz, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nz, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            nn.Linear(1024, img_dim),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

In [14]:
class VanillaDiscriminator(nn.Module):
    def __init__(self, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)

In [15]:
img_dim = 3 * image_size * image_size
nz = 100
netG = VanillaGenerator(nz, img_dim).to(device)
netD = VanillaDiscriminator(img_dim).to(device)

print(netG)
print(netD)


VanillaGenerator(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=512, out_features=1024, bias=True)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Linear(in_features=1024, out_features=3072, bias=True)
    (7): Tanh()
  )
)
VanillaDiscriminator(
  (net): Sequential(
    (0): Linear(in_features=3072, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=256, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


In [8]:
criterion = nn.BCELoss()

beta1 = 0.5

optimizerG = torch.optim.Adam(netG.parameters(), lr=2e-4, betas=(beta1, 0.999))
optimizerD = torch.optim.Adam(netD.parameters(), lr=1e-4, betas=(beta1, 0.999))

fixed_noise = torch.randn(64, nz, device=device)

G_losses = []
D_losses = []
real_scores = []
fake_scores = []

os.makedirs("outputs", exist_ok=True)

In [ ]:
from torchvision.utils import save_image

num_epochs = 300

best_d_loss = float("inf")  

for epoch in range(num_epochs):

    epoch_loss_g = 0.0
    epoch_loss_d = 0.0
    epoch_real = 0.0
    epoch_fake = 0.0

    for i, (real_images, _) in enumerate(
        tqdm(train_dl, desc=f"Epoch [{epoch+1}/{num_epochs}]")
    ):

        # --------------------
        # Move data to GPU
        # --------------------
        real_images = real_images.to(device, non_blocking=True)
        b_size = real_images.size(0)

        if epoch == 0 and i == 0:
            print("CUDA available:", torch.cuda.is_available())
            print("Tensor device:", real_images.device)
            print("GPU memory:",
                  torch.cuda.memory_allocated() // (1024**2), "MB")

        # Label smoothing
        real_labels = torch.full((b_size,), 0.9, device=device)
        fake_labels = torch.zeros(b_size, device=device)

        # --------------------
        # Train Discriminator
        # --------------------
        netD.zero_grad(set_to_none=True)

        real_flat = real_images.view(b_size, -1)
        real_out = netD(real_flat)
        d_loss_real = criterion(real_out, real_labels)

        noise = torch.randn(b_size, nz, device=device)
        fake_images = netG(noise)
        fake_out = netD(fake_images.detach())
        d_loss_fake = criterion(fake_out, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizerD.step()

        # --------------------
        # Train Generator
        # --------------------
        netG.zero_grad(set_to_none=True)

        fake_out = netD(fake_images)
        g_loss = criterion(fake_out, torch.ones(b_size, device=device))
        g_loss.backward()
        optimizerG.step()

        # --------------------
        # Logging
        # --------------------
        epoch_loss_d += d_loss.item()
        epoch_loss_g += g_loss.item()
        epoch_real += real_out.mean().item()
        epoch_fake += fake_out.mean().item()

    # Epoch averages
    n = len(train_dl)

    epoch_loss_d /= n
    epoch_loss_g /= n
    epoch_real /= n
    epoch_fake /= n

    D_losses.append(epoch_loss_d)
    G_losses.append(epoch_loss_g)
    real_scores.append(epoch_real)
    fake_scores.append(epoch_fake)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"loss_g: {epoch_loss_g:.4f}, "
        f"loss_d: {epoch_loss_d:.4f}, "
        f"real_score: {epoch_real:.4f}, "
        f"fake_score: {epoch_fake:.4f}"
    )

    # --------------------
    # Save images (fixed version)
    # --------------------
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).view(-1, 3, image_size, image_size)
            fake = fake.detach().cpu()

        save_image(
            fake,
            f"outputs/vanilla_{epoch+1:03d}.png",
            nrow=8,
            normalize=True
        )

    # --------------------
    # Save BEST discriminator
    # --------------------
    if epoch_loss_d < best_d_loss:
        best_d_loss = epoch_loss_d
        torch.save(netD.state_dict(), "checkpoints/best_vanilla_discriminator.pth")
        print("Saved new best vanilla discriminator")


# --------------------
# Final model save
# --------------------
torch.save(netG.state_dict(), "checkpoints/vanilla_generator_final.pth")
torch.save(netD.state_dict(), "checkpoints/vanilla_discriminator_final.pth")

Epoch [1/300]:   0%|          | 0/24 [00:00<?, ?it/s]

CUDA available: True
Tensor device: cuda:0
GPU memory: 23 MB


Epoch [1/300]: 100%|██████████| 24/24 [00:29<00:00,  1.24s/it]


Epoch [1/300] loss_g: 0.6848, loss_d: 1.3776, real_score: 0.6560, fake_score: 0.5074
Saved new best vanilla discriminator


Epoch [2/300]: 100%|██████████| 24/24 [00:34<00:00,  1.43s/it]


...

Epoch [299/300]: 100%|██████████| 24/24 [00:25<00:00,  1.05s/it]


Epoch [299/300] loss_g: 0.9994, loss_d: 1.3306, real_score: 0.4765, fake_score: 0.3817


Epoch [300/300]: 100%|██████████| 24/24 [00:24<00:00,  1.03s/it]

Epoch [300/300] loss_g: 1.0049, loss_d: 1.3399, real_score: 0.4679, fake_score: 0.3803


: 

In [16]:
torch.save(netD.state_dict(), "streamlit_app/models/vanilla_discriminator.pth")
torch.save(netG.state_dict(), "streamlit_app/models/vanilla_generator.pth")